# Self-Attention on different types of data

Installing all the required packages

In [ ]:
pip install bertviz transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 3.2 MB/s eta 0:00:00


In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import plotly.graph_objects as go
import pandas as pd
import seaborn as sns
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

In [ ]:
# ── Setting up the configuration ───────────────────────────────────────────────
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# ── Colors ───────────────────────────────────────────────────────────────────
AZUL_OSCURO   = "#1F3A5F"
AZUL_CLARO    = "#5DADE2"
VERDE_OSCURO  = "#1E8449"
VERDE_CLARO   = "#58D68D"
ROJO_OSCURO   = "#922B21"
ROJO_CLARO    = "#EC7063"
NARANJA       = "#E67E22"
AMARILLO      = "#F4D03F"
MORADO        = "#7D3C98"
GRIS_OSCURO   = "#566573"
DARK_BG       = '#0F0F1A'
PANEL_BG      = '#1A1A2E'
ACCENT1       = '#E94560'
ACCENT2       = '#1a5276'
ACCENT3       = '#16213E'
TEXT_CLR      = '#E0E0E0'
GRID_CLR      = '#2A2A45'
NORMAL_CLR    = '#FF1500'
GOLD          = '#FFD700'
CYAN          = '#00D4FF'
BLANCO        = '#FFFFFF'

# Graphic representation of Self-attention (Text)

In [ ]:
from transformers import AutoTokenizer, AutoModel
from bertviz import head_view, model_view, neuron_view
import torch

# 1. Choose a classic model (BERT-base)
model_name = "bert-base-uncased"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, output_attentions=True).to(device)
model.eval()  # Disable dropout: attention weights must be deterministic.

# 2. Prepare your exact text sequence
text = "All data in deep learning must be represented as vectors."

# 3. Tokenize including attention_mask (key for masking padding in softmax)
inputs = tokenizer(text, return_tensors="pt").to(device)

# 4. Forward pass without gradients
with torch.inference_mode():
    outputs = model(**inputs)

attention = outputs.attentions  # tuple: (num_layers,) each (batch, num_heads, seq_len, seq_len)

if not attention:
    print("Lista de atención vacía")
else:
    n_layers = len(attention)
    n_heads = attention[0].shape[1]
    seq_len = attention[0].shape[-1]
    print(f"{n_layers} capas × {n_heads} cabezas de atención por capa, seq_len={seq_len}")
    # This corresponds exactly to Multi-Head Attention:
    # each head learns its own Q, K, V projection,
    # and softmax(QK^T / sqrt(d_k)) V is computed in parallel for each head.

# 5. Tokens for visualization labels
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


12 capas × 12 cabezas de atención por capa, seq_len=13


In [ ]:
from IPython.display import display, HTML

display(HTML(f"""
<h2 style="color:#5DADE2;">
BERT Multi-Head Self-Attention
</h2>

<table>
<tr><td><b>Model</b></td><td>{model_name}</td></tr>
<tr><td><b>Layers</b></td><td>{n_layers}</td></tr>
<tr><td><b>Heads</b></td><td>{n_heads}</td></tr>
<tr><td><b>Sequence length</b></td><td>{seq_len}</td></tr>
<tr><td><b>Device</b></td><td>{device}</td></tr>
</table>
"""))

Model,bert-base-uncased
Layers,12
Heads,12
Sequence length,13
Device,cpu


In [ ]:
# 6. Interactive visualization (move to CPU if the model ran on GPU)
attention_cpu = tuple(layer.cpu() for layer in attention)
head_view(attention_cpu, tokens)

<IPython.core.display.Javascript object>

# Graphic representation of Self-attention (Images)

In [ ]:
import torch
import numpy as np
import requests
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import ipywidgets as widgets
from IPython.display import display
from transformers import ViTImageProcessor, ViTForImageClassification


def compute_all_attentions(image, model, feature_extractor):
    """
    Runs the forward pass only once and returns all attention maps
    (num_layers, num_heads, seq_len, seq_len) together with the processed image.
    This avoids recomputing Q, K, and V every time the user interacts.
    """
    inputs = feature_extractor(images = image, return_tensors = "pt")
    pixel_values = inputs.pixel_values

    with torch.no_grad():
        outputs = model(pixel_values, output_attentions = True)

    # attentions: tuple of (num_layers,) tensors with shape
    # [1, num_heads, seq_len, seq_len]
    attentions = torch.stack(outputs.attentions).squeeze(1)
    return attentions, pixel_values[0]


def plot_interactive_attention(original_image, model, feature_extractor):
    grid_size   = 14      # 224 / 16
    patch_size  = 16
    num_layers  = model.config.num_hidden_layers
    num_heads   = model.config.num_attention_heads

    print("Computing attention (single forward pass)...")
    attentions, processed_pixel = compute_all_attentions(
        original_image, model, feature_extractor
    )
    img_disp = original_image.resize((224, 224))

    # --- Control widgets ---
    layer_slider = widgets.IntSlider(
        value = 0,
        min = 0,
        max = num_layers - 1,
        description = "Layer:",
        continuous_update = False
    )

    head_slider = widgets.IntSlider(
        value = 0,
        min = 0,
        max = num_heads - 1,
        description = "Head:",
        continuous_update = False
    )

    row_slider = widgets.IntSlider(
        value = 0,
        min = 0,
        max = grid_size - 1,
        description = "Patch row:",
        continuous_update = False
    )

    col_slider = widgets.IntSlider(
        value = 0,
        min = 0,
        max = grid_size - 1,
        description = "Patch column:",
        continuous_update = False
    )

    mode_toggle = widgets.ToggleButtons(
        options = [
            ("Global Attention ([CLS] token)", "cls"),
            ("Single Patch Query", "patch")
        ],
        description = "Mode:"
    )

    out = widgets.Output()

    def get_attn_grid(layer, head, mode, row, col):
        # [197, 197]: A_ij = softmax(QK^T/sqrt(d_k))_ij
        attn_head = attentions[layer, head]

        if mode == "cls":
            # Attention from the [CLS] token (query)
            # to every image patch (keys)
            query_idx    = 0
            title_suffix = "(Query = [CLS])"
        else:
            query_idx = row * grid_size + col + 1  # +1 because CLS is the first token
            title_suffix = f"(Query = patch [{row}, {col}])"

        # Ignore the CLS key and keep only the 196 image patches
        vec = attn_head[query_idx, 1:]
        return vec.reshape(grid_size, grid_size).numpy(), title_suffix

    def redraw(*_):
        with out:
            out.clear_output(wait = True)

            attn_grid, title_suffix = get_attn_grid(
                layer_slider.value,
                head_slider.value,
                mode_toggle.value,
                row_slider.value,
                col_slider.value
            )

            fig, (ax1, ax2) = plt.subplots(
                1, 2,
                figsize = (14, 6),
                facecolor = DARK_BG,
                constrained_layout = True
            )

            # ------------------------------------------------------------------
            # Original image
            # ------------------------------------------------------------------

            ax1.set_facecolor(PANEL_BG)

            ax1.imshow(img_disp)

            ax1.set_title(
                "Original Image\n14×14 Vision Transformer Patches",
                color = BLANCO,
                pad = 10
            )

            ax1.axis("off")

            for i in range(1, grid_size):
                ax1.axhline(
                    i * patch_size,
                    color = GRID_CLR,
                    linewidth = 0.5,
                    alpha = 0.6
                )
                ax1.axvline(
                    i * patch_size,
                    color = GRID_CLR,
                    linewidth = 0.5,
                    alpha = 0.6
                )

            if mode_toggle.value == "patch":

                r = row_slider.value
                c = col_slider.value

                ax1.add_patch(

                    plt.Rectangle(

                        (c * patch_size, r * patch_size),

                        patch_size,

                        patch_size,

                        fill = False,

                        edgecolor = GOLD,

                        linewidth = 2.5,

                    )

                )

            # ------------------------------------------------------------------
            # Attention map
            # ------------------------------------------------------------------

            ax2.set_facecolor(PANEL_BG)

            ax2.imshow(img_disp, alpha = 0.35)

            heatmap = ax2.imshow(

                attn_grid,

                cmap = "magma",

                interpolation = "bicubic",

                extent = [0, 224, 224, 0],

                alpha = 0.92,

            )

            ax2.set_title(

                f"Self-Attention\nLayer {layer_slider.value} • Head {head_slider.value}\n{title_suffix}",

                color = BLANCO,

                pad = 10

            )

            ax2.axis("off")

            cbar = fig.colorbar(

                heatmap,

                ax = ax2,

                fraction = 0.046,

                pad = 0.03

            )

            cbar.set_label(

                "Attention Weight",

                color = TEXT_CLR

            )

            cbar.ax.tick_params(colors = TEXT_CLR)

            plt.setp(
                cbar.ax.get_yticklabels(),
                color = TEXT_CLR
            )

            for ax in (ax1, ax2):
                for spine in ax.spines.values():
                    spine.set_color(GRID_CLR)

            plt.show()

    # Show or hide the row/column sliders depending on the selected mode
    def on_mode_change(change):
        show = change["new"] == "patch"
        row_slider.layout.display = "" if show else "none"
        col_slider.layout.display = "" if show else "none"
        redraw()

    mode_toggle.observe(on_mode_change, names="value")

    for w in (layer_slider, head_slider, row_slider, col_slider):
        w.observe(redraw, names = "value")

    row_slider.layout.display = "none"
    col_slider.layout.display = "none"

    controls = widgets.VBox(
        [mode_toggle, layer_slider, head_slider, row_slider, col_slider]
    )
    display(controls, out)
    redraw()


# --- RUN WITH A SAMPLE IMAGE ---
if __name__ == "__main__":
    model_name = "google/vit-base-patch16-224"

    print(f"Loading model {model_name}...")
    feature_extractor = ViTImageProcessor.from_pretrained(model_name)
    model = ViTForImageClassification.from_pretrained(
        model_name,
        output_attentions = True
    )
    model.eval()  # Disable dropout for deterministic attention weights

    url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/cat.jpg" # Changeable

    print("Downloading sample image...")
    original_image = Image.open(
        requests.get(url, stream = True).raw
    ).convert("RGB")

    plot_interactive_attention(
        original_image,
        model,
        feature_extractor
    )

Loading model google/vit-base-patch16-224...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Computing attention (single forward pass)...


Output()

# Graphic representation of Self-attention (Audios)

Attention Mechanism Visualization for Audio Models
==================================================

This script loads an audio Transformer model (Wav2Vec2) from the Hugging Face Hub,extracts the internal attention matrices (self-attention),
and displays them over the input audio spectrogram.

It is designed to run in Jupyter (it uses ipywidgets for the interactive
layer/head selector), but it also works as a regular Python script.
In that case, it generates and saves the figures to disk.

**Requirements:**
    pip install torch transformers datasets librosa matplotlib ipywidgets soundfile

**Model used: facebook/wav2vec2-base-960h**
- Transformer encoder with 12 layers and 12 attention heads per layer.
- output_attentions=True returns a tuple (one entry per layer)
      containing tensors with shape [batch, n_heads, seq_len, seq_len].

Sample audio: dataset "hf-internal-testing/librispeech_asr_dummy"\
    (the same type of source used in the official Wav2Vec2 examples
    from the Hugging Face documentation).

In [ ]:
import librosa
import librosa.display
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from datasets import load_dataset

**Dataset demostration**

In [ ]:
audioDataset = "hf-internal-testing/librispeech_asr_dummy" # Dataset from Hugging Face

In [ ]:
import IPython.display as ipd
from datasets import load_dataset

# The variable 'audioDataset' is expected to be defined from cell H-Nx7FNN2WJs
# The function 'load_dataset' is expected to be imported from cell vtaJnsZ62Zdt
dataset = load_dataset(audioDataset, "clean", split = "validation")
sample = dataset[0]

ipd.Audio(sample["audio"]["array"], rate = sample["audio"]["sampling_rate"], autoplay = False)

clean/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 9.19MB            

clean/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/73 [00:00<?, ? examples/s]

-----

In [ ]:
# -----------------------------------------------------------------------
# Graphic configuration
# -----------------------------------------------------------------------

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

plt.rcParams.update({

    "figure.facecolor" : DARK_BG,
    "axes.facecolor"   : PANEL_BG,
    "savefig.facecolor": DARK_BG,

    "axes.edgecolor"   : GRID_CLR,
    "axes.labelcolor"  : TEXT_CLR,

    "xtick.color"      : TEXT_CLR,
    "ytick.color"      : TEXT_CLR,

    "text.color"       : TEXT_CLR,

    "grid.color"       : GRID_CLR,
    "grid.alpha"       : 0.35,

    "font.size"        : 12,
    "axes.titlesize"   : 15,
    "axes.labelsize"   : 12,
    "figure.titlesize" : 18,

    "axes.titleweight" : "bold",
})

In [ ]:
# -----------------------------------------------------------------------
# 1. Load the model and processor from Hugging Face
# -----------------------------------------------------------------------
MODEL_NAME = "facebook/wav2vec2-base-960h"

print(f"Loading model '{MODEL_NAME}' from the Hugging Face Hub...")
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model     = Wav2Vec2Model.from_pretrained(MODEL_NAME, output_attentions = True)
model.eval()

# -----------------------------------------------------------------------
# 2. Load a sample audio file from a Hub dataset
# -----------------------------------------------------------------------
print("Downloading sample audio (LibriSpeech dummy)...")
dataset = load_dataset(
    audioDataset, "clean", split = "validation"
)
sample        = dataset[0]
audio_array   = sample["audio"]["array"].astype(np.float32)
sampling_rate = sample["audio"]["sampling_rate"]  # 16 kHz, required by Wav2Vec2

# Crop the audio to about 4 seconds so the attention matrix is easier to read
max_seconds = 4
max_samples = int(max_seconds * sampling_rate)
audio_array = audio_array[:max_samples]

print(f"Loaded audio: {len(audio_array) / sampling_rate:.2f} s at {sampling_rate} Hz")

# -----------------------------------------------------------------------
# 3. Forward pass to extract the attention maps
# -----------------------------------------------------------------------
inputs = processor(audio_array, sampling_rate=sampling_rate, return_tensors = "pt")

with torch.no_grad():
    outputs = model(**inputs, output_attentions = True)

# outputs.attentions: tuple with n_layers elements
# each element: [batch=1, n_heads, seq_len, seq_len]
attentions  = outputs.attentions
n_layers    = len(attentions)
n_heads     = attentions[0].shape[1]
seq_len     = attentions[0].shape[-1]

print(
    f"Extracted attention maps: {n_layers} layers × {n_heads} heads, "
    f"sequence length of {seq_len} steps "
    f"(convolutional frames, about 20 ms each)"
)

# -----------------------------------------------------------------------
# 4. Visualization utilities
# -----------------------------------------------------------------------

def frame_times(n_frames, total_seconds):
    """Estimate the time (in seconds) of each output frame from the
    convolutional encoder, assuming uniform spacing."""
    return np.linspace(0, total_seconds, n_frames)


def plot_attention_heatmap(layer, head, ax = None, average_heads = False):
    """Draw the attention heatmap [seq_len × seq_len] for a given
    layer and head (or averaged across all heads)."""

    if average_heads:
        attn = attentions[layer][0].mean(dim = 0).numpy()
        title = f"Layer {layer} • Average of {n_heads} Heads"
    else:
        attn = attentions[layer][0, head].numpy()
        title = f"Layer {layer} • Head {head}"

    if ax is None:
        fig, ax = plt.subplots(
            figsize   = (7, 6),
            facecolor = DARK_BG
        )

    im = ax.imshow(
        attn,
        cmap = "mako",
        origin = "lower",
        aspect = "auto",
        interpolation = "bicubic",
    )

    ax.set_title(title, color = BLANCO, pad = 12)
    ax.set_xlabel("Attended Position (Frame)")
    ax.set_ylabel("Query Position (Frame)")
    ax.grid(False)

    for spine in ax.spines.values():
        spine.set_color(GRID_CLR)

    cbar = plt.colorbar(
        im,
        ax = ax,
        fraction = 0.045,
        pad = 0.03
    )

    cbar.set_label("Attention Weight", color = TEXT_CLR)
    cbar.ax.yaxis.set_tick_params(color = TEXT_CLR)

    plt.setp(
        cbar.ax.get_yticklabels(),
        color = TEXT_CLR
    )
    return ax


def plot_attention_over_spectrogram(layer, head, query_frame = None):
    """Overlay the attention values on the real audio spectrogram,
    showing how much attention each frame receives from a selected
    query frame."""

    attn = attentions[layer][0, head].numpy()

    if query_frame is None:
        attn_profile = attn.mean(axis = 0)
        subtitle = "Average Attention"

    else:
        attn_profile = attn[query_frame]
        subtitle = f"Attention from Frame {query_frame}"

    times = frame_times(
        seq_len,
        len(audio_array) / sampling_rate
    )

    fig, (ax1, ax2) = plt.subplots(
        2, 1,
        figsize = (12, 7),
        sharex = True,
        gridspec_kw = {
            "height_ratios": [3.5, 1]
        },
        facecolor = DARK_BG,
        constrained_layout = True,

    )

    S = librosa.feature.melspectrogram(
        y = audio_array,
        sr = sampling_rate,
        n_mels = 64

    )

    S_db = librosa.power_to_db(
        S,
        ref = np.max
    )

    librosa.display.specshow(
        S_db,
        sr = sampling_rate,
        x_axis = "time",
        y_axis = "mel",
        cmap = "magma",
        ax = ax1,
    )

    ax1.set_title(
        f"Spectrogram with Self-Attention\nLayer {layer} • Head {head}",
        fontsize = 16,
        color = BLANCO,
        pad = 12,
    )

    ax1.text(
        0.01,
        0.96,
        subtitle,
        transform = ax1.transAxes,
        color = GOLD,
        fontsize = 11,
        bbox = dict(
            facecolor = ACCENT3,
            edgecolor = GRID_CLR,
            alpha = 0.85,
            boxstyle = "round"
        )
    )

    ax2.plot(
        times,
        attn_profile,
        color = CYAN,
        linewidth = 2.5,
    )

    ax2.fill_between(
        times,
        attn_profile,
        color = CYAN,
        alpha = 0.35,
    )

    ax2.scatter(
        times,
        attn_profile,
        s = 10,
        color = GOLD,
        alpha = 0.6,
    )

    ax2.set_ylabel("Weight")
    ax2.set_xlabel("Time (s)")

    ax2.grid(
        True,
        linestyle = "--",
        alpha = 0.30
    )

    ax2.set_ylim(0, attn_profile.max() * 1.10)

    for ax in [ax1, ax2]:
        ax.set_facecolor(PANEL_BG)
        for spine in ax.spines.values():
            spine.set_color(GRID_CLR)

    return fig

# -----------------------------------------------------------------------
# 5. Interactive mode (Jupyter) vs. script mode (save figures to disk)
# -----------------------------------------------------------------------

def _running_in_jupyter():
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except ImportError:
        return False


if _running_in_jupyter():
    import ipywidgets as widgets
    from IPython.display import display

    layer_slider = widgets.IntSlider(
        min         = 0,
        max         = n_layers - 1,
        value       = 0,
        description = "Layer"
    )

    head_slider = widgets.IntSlider(
        min         = 0,
        max         = n_heads - 1,
        value       = 0,
        description = "Head"
    )

    average_toggle = widgets.Checkbox(
        value       = False,
        description = "Average Heads"
    )

    def _update(layer, head, average_heads):
      fig, ax = plt.subplots(figsize=(6, 5))
      plot_attention_heatmap(
            layer,
            head,
            ax = ax,
            average_heads = average_heads,
      )
      plt.show()

      plot_attention_over_spectrogram(layer, head)
      plt.show()

    print("Interactive widget: move the sliders to explore different layers and heads.")

    widgets.interact(
        _update,
        layer         = layer_slider,
        head          = head_slider,
        average_heads = average_toggle,
    )

else:
    print("Jupyter environment not detected: generating sample figures and saving them to disk...")

    # Heatmap for a representative layer and head
    fig1, ax1 = plt.subplots(
      figsize = (8, 6),
      facecolor = DARK_BG
    )

    plot_attention_heatmap(layer = 6, head = 0, ax = ax1)
    fig1.tight_layout()

    # Heatmap averaged across all heads in the last layer
    fig2, ax2 = plt.subplots(
      figsize = (8, 6),
      facecolor = DARK_BG
    )

    plot_attention_heatmap(
        layer         = n_layers - 1,
        head          = 0,
        ax            = ax2,
        average_heads = True,
    )
    fig2.tight_layout()

    # Attention overlaid on the real spectrogram
    fig3 = plot_attention_over_spectrogram(layer = 6, head = 0)

Loading model 'facebook/wav2vec2-base-960h' from the Hugging Face Hub...


Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded audio: 4.00 s at 16000 Hz
Extracted attention maps: 12 layers × 12 heads, sequence length of 199 steps (convolutional frames, about 20 ms each)
Interactive widget: move the sliders to explore different layers and heads.


interactive(children=(IntSlider(value=0, description='Layer', max=11), IntSlider(value=0, description='Head', …